In [1]:
#Instalo PySpark y astrapy
!pip install -q pyspark astrapy

#Clono el repo del proyecto, donde ya tengo el zip de la data
!rm -rf /content/cloud-provider-analytics
!git clone -q https://github.com/damiann8n/cloud-provider-analytics.git /content/cloud-provider-analytics

#Descomprimo. El zip ya trae la carpeta q' necesito
!rm -rf /content/datalake
!unzip -q -o /content/cloud-provider-analytics/data/cloud_provider_challenge_dataset_v1.zip -d /content

#Reviso que este todo: los 7 CSV y la carpeta de eventos (x las dudas)
!ls /content/datalake/landing
!ls /content/datalake/landing/usage_events_stream | wc -l

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 9.19.1 which is incompatible.
billing_monthly.csv    nps_surveys.csv	    usage_events_stream
customers_orgs.csv     resources.csv	    users.csv
marketing_touches.csv  support_tickets.csv
120


In [2]:
#Configuracion del pipeline. La dejo en un solo lugar para no tener valores sueltos por el notebook
import json, os

CONFIG = {
    "landing": "/content/datalake/landing",
    "bronze": "/content/datalake/bronze",
    "silver": "/content/datalake/silver",
    "gold": "/content/datalake/gold",
    "quarantine": "/content/datalake/quarantine",
    "checkpoints": "/content/datalake/_checkpoints",
    "watermark": "60 days",          #ver D-04 y D-05
    "max_files_per_trigger": 10,     #10 archivos por micro-lote, son 12 micro-lotes
    "costo_minimo": -0.01,           #regla de calidad de la consigna
    "astra_keyspace": "default_keyspace",
    "tabla_diaria": "org_daily_usage_by_service",
    "tabla_top14": "top_services_14d"
}

#Guardo la config tambien como archivo, asi queda externalizada como pide la consigna
os.makedirs("/content/config", exist_ok=True)
with open("/content/config/config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

print(json.dumps(CONFIG, indent=2))

{
  "landing": "/content/datalake/landing",
  "bronze": "/content/datalake/bronze",
  "silver": "/content/datalake/silver",
  "gold": "/content/datalake/gold",
  "quarantine": "/content/datalake/quarantine",
  "checkpoints": "/content/datalake/_checkpoints",
  "watermark": "60 days",
  "max_files_per_trigger": 10,
  "costo_minimo": -0.01,
  "astra_keyspace": "default_keyspace",
  "tabla_diaria": "org_daily_usage_by_service",
  "tabla_top14": "top_services_14d"
}


In [3]:
#Creo la sesion de Spark, igual que en el notebook de clase
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = (SparkSession.builder
         .appName("cloud-provider-analytics")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         #Zona horaria en UTC, como dijo el profesor
         .config("spark.sql.session.timeZone", "UTC")
         #Rescribo Silver y Gold, pero solo piso las particiones (dias) que cambian
         .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
         .getOrCreate())

print("Chequeo de Spark", spark.version)

Chequeo de Spark 4.0.5


In [4]:
#Defino un esquema para cada CSV, y así evito utilizar inferSchema, controlando los tipos sin dejar que Spark los deduzca
esquemas_csv = {
    "customers_orgs": StructType([
        StructField("org_id", StringType()), StructField("org_name", StringType()),
        StructField("industry", StringType()), StructField("hq_region", StringType()),
        StructField("plan_tier", StringType()), StructField("is_enterprise", BooleanType()),
        StructField("signup_date", DateType()), StructField("sales_rep", StringType()),
        StructField("lifecycle_stage", StringType()), StructField("marketing_source", StringType()),
        StructField("nps_score", DoubleType())]),
    "users": StructType([
        StructField("user_id", StringType()), StructField("org_id", StringType()),
        StructField("email", StringType()), StructField("role", StringType()),
        StructField("active", BooleanType()), StructField("created_at", DateType()),
        StructField("last_login", DateType())]),
    "resources": StructType([
        StructField("resource_id", StringType()), StructField("org_id", StringType()),
        StructField("service", StringType()), StructField("region", StringType()),
        StructField("created_at", DateType()), StructField("state", StringType()),
        StructField("tags_json", StringType())]),
    "support_tickets": StructType([
        StructField("ticket_id", StringType()), StructField("org_id", StringType()),
        StructField("category", StringType()), StructField("severity", StringType()),
        StructField("created_at", DateType()), StructField("resolved_at", DateType()),
        StructField("csat", DoubleType()), StructField("sla_breached", BooleanType())]),
    "marketing_touches": StructType([
        StructField("touch_id", StringType()), StructField("org_id", StringType()),
        StructField("campaign", StringType()), StructField("channel", StringType()),
        StructField("timestamp", DateType()), StructField("clicked", BooleanType()),
        StructField("converted", BooleanType())]),
    "nps_surveys": StructType([
        StructField("org_id", StringType()), StructField("survey_date", DateType()),
        StructField("nps_score", DoubleType()), StructField("comment", StringType())]),
    "billing_monthly": StructType([
        StructField("invoice_id", StringType()), StructField("org_id", StringType()),
        StructField("month", DateType()), StructField("subtotal", DoubleType()),
        StructField("credits", DoubleType()), StructField("taxes", DoubleType()),
        StructField("currency", StringType()), StructField("exchange_rate_to_usd", DoubleType())])
}

print("Esquemas definidos:", list(esquemas_csv.keys()))

Esquemas definidos: ['customers_orgs', 'users', 'resources', 'support_tickets', 'marketing_touches', 'nps_surveys', 'billing_monthly']


In [5]:
#Clave natural de cada tabla, para sacar duplicados si los hubiera.
#nps_surveys no tiene una clave unica, ahi saco solo las filas repetidas enteras
claves = {
    "customers_orgs": ["org_id"], "users": ["user_id"], "resources": ["resource_id"],
    "support_tickets": ["ticket_id"], "marketing_touches": ["touch_id"],
    "nps_surveys": None, "billing_monthly": ["invoice_id"]
}

conteos_bronze = {}

for tabla, esquema in esquemas_csv.items():
    #Leo el CSV de Landing con su esquema
    df = (spark.read
          .option("header", True)
          .schema(esquema)
          .csv(f"{CONFIG['landing']}/{tabla}.csv"))

    #Agrego las columnas tecnicas: cuando lo cargue y de que archivo vino
    df = (df.withColumn("ingest_ts", F.current_timestamp())
            .withColumn("source_file", F.col("_metadata.file_path")))

    filas_leidas = df.count()

    #Saco duplicados por la clave natural
    if claves[tabla]:
        df = df.dropDuplicates(claves[tabla])
    else:
        df = df.dropDuplicates([c for c in df.columns if c not in ("ingest_ts", "source_file")])

    filas_bronze = df.count()

    #Escribo en Bronze. Solo particiono billing por mes (D-03); los maestros son chicos.
    #overwrite: si vuelvo a correr, piso la tabla y no se duplica nada
    escritor = df.coalesce(1).write.mode("overwrite")
    if tabla == "billing_monthly":
        escritor = escritor.partitionBy("month")
    escritor.parquet(f"{CONFIG['bronze']}/{tabla}")

    conteos_bronze[tabla] = (filas_leidas, filas_bronze)
    #imprimo para analizar que todo esté correctamente codificado y funcione
    print(f"{tabla:20s} leidas={filas_leidas:5d}  en bronze={filas_bronze:5d}")

customers_orgs       leidas=   80  en bronze=   80
users                leidas=  800  en bronze=  800
resources            leidas=  400  en bronze=  400
support_tickets      leidas= 1000  en bronze= 1000
marketing_touches    leidas= 1500  en bronze= 1500
nps_surveys          leidas=   92  en bronze=   92
billing_monthly      leidas=  240  en bronze=  240


In [6]:
#Esquema explicito de los eventos. Incluye los campos de v1 y de v2:
#si un evento v1 no trae carbon_kg o genai_tokens, Spark los deja en null
esquema_eventos = StructType([
    StructField("event_id", StringType()),
    StructField("timestamp", StringType()),       #lo convierto a fecha despues (no olvidar)
    StructField("org_id", StringType()),
    StructField("resource_id", StringType()),
    StructField("service", StringType()),
    StructField("region", StringType()),
    StructField("metric", StringType()),
    StructField("value", StringType()),           #llega a veces como numero y a veces como texto, luego analizo
    StructField("unit", StringType()),
    StructField("cost_usd_increment", DoubleType()),
    StructField("schema_version", IntegerType()),
    StructField("carbon_kg", DoubleType()),       #solo v2
    StructField("genai_tokens", LongType()),      #solo v2 y servicio genai
    StructField("_corrupt_record", StringType())  #lineas que no se pueden leer, como en el ejemplo de clase
])
#chequeo
print(esquema_eventos.simpleString())

struct<event_id:string,timestamp:string,org_id:string,resource_id:string,service:string,region:string,metric:string,value:string,unit:string,cost_usd_increment:double,schema_version:int,carbon_kg:double,genai_tokens:bigint,_corrupt_record:string>


In [7]:
#VOY CON EL SOURCE: leo la carpeta de eventos como un stream de archivos
eventos_stream = (spark.readStream
    .schema(esquema_eventos)
    .option("mode", "PERMISSIVE")                            #no sé cae si una linea esta rota
    .option("columnNameOfCorruptRecord", "_corrupt_record")  #lo guardo en esta columna
    .option("maxFilesPerTrigger", CONFIG["max_files_per_trigger"])
    .json(f"{CONFIG['landing']}/usage_events_stream"))

#HAGO LA TRANSFORMACION: convierto la fecha y agrego las columnas tecnicas
eventos_bronze = (eventos_stream
    .withColumn("event_ts", F.to_timestamp("timestamp"))
    .withColumn("event_date", F.to_date("event_ts"))
    .withColumn("ingest_ts", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
    #Watermark de 60 dias (D-05) y quito duplicados por event_id
                  #esto es de la primera vuelta
    #.withWatermark("event_ts", CONFIG["watermark"])
    #.dropDuplicatesWithinWatermark(["event_id"]))
    #SEGUNDA VUELTA:
    .withWatermark("event_ts", CONFIG["watermark"])
    .dropDuplicatesWithinWatermark(["event_id"])
    #Junto los eventos de cada dia en una misma particion: asi escribe 1 archivo por dia
    #en cada micro-lote, en vez de 8 (evito el problema de archivos chicos de la primera corrida)
    .repartition("event_date"))
#SINK parquet, modo append, trigger availableNow (así proceso todo lo que hay y termina la joda)
query_bronze = (eventos_bronze.writeStream
    .format("parquet")
    .outputMode("append")
    .partitionBy("event_date")
    .option("path", f"{CONFIG['bronze']}/usage_events")
    .option("checkpointLocation", f"{CONFIG['checkpoints']}/bronze_usage_events")
    .trigger(availableNow=True)
    .start())

query_bronze.awaitTermination()

#Cuento lo que quedó en Bronze
bronze_eventos = spark.read.parquet(f"{CONFIG['bronze']}/usage_events")
#chequeo, otra vez. siempre:
print("Eventos en Bronze:", bronze_eventos.count())
print("Micro-lotes procesados:", len(query_bronze.recentProgress))

Eventos en Bronze: 43200
Micro-lotes procesados: 12


In [8]:
def validar_eventos(df):
    #Convierto value a numero. try_cast devuelve null si no puede, en vez de cortar el programa y que se caiga
    df = df.withColumn("value_num", F.expr("try_cast(value as double)"))

    #Reviso las 3 reglas de la consigna del Parcial II y guardo el motivo del primer problema que encuentro.
    #coalesce(..., False), para que un null no rompa la comparación
    regla1 = F.col("event_id").isNull()
    regla2 = F.coalesce(F.col("cost_usd_increment") < CONFIG["costo_minimo"], F.lit(False))
    regla3 = F.col("value_num").isNotNull() & F.col("unit").isNull()

    df = df.withColumn("motivo",
        F.when(F.col("_corrupt_record").isNotNull(), "registro_corrupto")
         .when(regla1, "event_id_nulo")
         .when(regla2, "costo_negativo")
         .when(regla3, "unit_nulo_con_value")
         .otherwise(None))
    return df

#Pruebo la funcion con todo lo que hay en Bronze
prueba = validar_eventos(spark.read.parquet(f"{CONFIG['bronze']}/usage_events"))
#muestro en pantalla el resultado
prueba.groupBy("motivo").count().orderBy("count", ascending=False).show()

+-------------------+-----+
|             motivo|count|
+-------------------+-----+
|               NULL|40956|
|unit_nulo_con_value| 2033|
|     costo_negativo|  211|
+-------------------+-----+



In [9]:
#Leo el maestro de organizaciones de Bronze para apuntar a los eventos
orgs = (spark.read.parquet(f"{CONFIG['bronze']}/customers_orgs")
        .select("org_id", "org_name", "industry", "plan_tier", "is_enterprise"))

def armar_silver(df_validos):
    df = (df_validos
        #Paso cada mtrica a su propia columna (c/features x evento)
        .withColumn("requests",         F.when(F.col("metric") == "requests", F.col("value_num")))
        .withColumn("cpu_hours",        F.when(F.col("metric") == "cpu_hours", F.col("value_num")))
        .withColumn("storage_gb_hours", F.when(F.col("metric") == "storage_gb_hours", F.col("value_num")))
        #Compatibilidad v1/v2: en v1 carbon_kg y genai_tokens quedan en null
        .withColumn("processed_ts", F.current_timestamp())
        #Mejoro con los datos de la organizacion
        .join(F.broadcast(orgs), on="org_id", how="left")
        .select("event_id", "event_ts", "event_date", "org_id", "org_name", "industry", "plan_tier",
                "is_enterprise", "resource_id", "service", "region", "metric", "value_num", "unit",
                "cost_usd_increment", "requests", "cpu_hours", "storage_gb_hours",
                "genai_tokens", "carbon_kg", "schema_version", "source_file", "ingest_ts", "processed_ts"))
    return df

#Pruebo con los validos de la celda anterior
silver_prueba = armar_silver(prueba.filter(F.col("motivo").isNull()))
print("Filas en Silver:", silver_prueba.count())
#chequear
print("Eventos sin organizacion:", silver_prueba.filter(F.col("org_name").isNull()).count())
silver_prueba.select("event_date", "org_name", "service", "metric", "requests", "cpu_hours",
                     "cost_usd_increment", "genai_tokens", "carbon_kg").show(5)

Filas en Silver: 40956
Eventos sin organizacion: 0
+----------+----------------+----------+----------------+--------+---------+------------------+------------+---------+
|event_date|        org_name|   service|          metric|requests|cpu_hours|cost_usd_increment|genai_tokens|carbon_kg|
+----------+----------------+----------+----------------+--------+---------+------------------+------------+---------+
|2025-08-04|    Omega Labs 9|  database|       cpu_hours|    NULL|   5.2258|            0.3102|        NULL| 0.001045|
|2025-08-04|     Apex Data 3|networking|storage_gb_hours|    NULL|     NULL|            0.0632|        NULL|  0.00158|
|2025-08-04|  Vertex Labs 22|   compute|storage_gb_hours|    NULL|     NULL|            0.1177|        NULL|  3.06E-4|
|2025-08-04|      Nova AI 74|networking|       cpu_hours|    NULL|   0.7227|            0.0056|        NULL|  1.45E-4|
|2025-08-04|Gamma Digital 20|   compute|storage_gb_hours|    NULL|     NULL|            0.8417|        NULL| 0.00224

In [10]:
#armo una función: las vengo usando poco y nada
def armar_gold(df_silver):
    #Grano: una fila por organizacion + dia + servicio (consigna, mart FinOps)
    return (df_silver
        .groupBy("org_id", "org_name", F.col("event_date").alias("usage_date"), "service")
        .agg(F.round(F.sum("cost_usd_increment"), 4).alias("daily_cost_usd"),
             F.sum("requests").alias("requests"),
             F.round(F.sum("cpu_hours"), 4).alias("cpu_hours"),
             F.round(F.sum("storage_gb_hours"), 4).alias("storage_gb_hours"),
             F.sum("genai_tokens").alias("genai_tokens"),
             F.round(F.sum("carbon_kg"), 6).alias("carbon_kg"),
             F.count("*").alias("eventos"))
        .withColumn("processed_ts", F.current_timestamp()))

#Pruebo con el Silver de la celda anterior
gold_prueba = armar_gold(silver_prueba)
#check
print("Filas en Gold:", gold_prueba.count())
gold_prueba.orderBy("org_id", "usage_date", "service").show(5)

Filas en Gold: 11050
+------------+-------------+----------+---------+--------------+--------+---------+----------------+------------+---------+-------+--------------------+
|      org_id|     org_name|usage_date|  service|daily_cost_usd|requests|cpu_hours|storage_gb_hours|genai_tokens|carbon_kg|eventos|        processed_ts|
+------------+-------------+----------+---------+--------------+--------+---------+----------------+------------+---------+-------+--------------------+
|org_0lvsnujz|Nova Cloud 45|2025-07-03|analytics|        8.8739|   116.0|   4.6168|         12.7808|        NULL|     NULL|      3|2026-11-25 00:20:...|
|org_0lvsnujz|Nova Cloud 45|2025-07-03| database|        0.5723|    NULL|   1.7515|         14.7891|        NULL|     NULL|      2|2026-11-25 00:20:...|
|org_0lvsnujz|Nova Cloud 45|2025-07-03|  storage|        5.5542|   255.0|     NULL|          4.6668|        NULL|     NULL|      3|2026-11-25 00:20:...|
|org_0lvsnujz|Nova Cloud 45|2025-07-04|analytics|        3.29

In [11]:
'''antes de la primera vuelta que me generó problemas con el tamaño
def procesar_lote(lote_df, lote_id):
    #Qué dias me trae este micro-lote?
    fechas = [r.event_date for r in lote_df.select("event_date").distinct().collect()]
    if not fechas:
        return

    #Leo desde Bronze TODOS los eventos de esos dias (no solo los del lote) y los valido
    eventos = (spark.read.parquet(f"{CONFIG['bronze']}/usage_events")
               .filter(F.col("event_date").isin(fechas)))
    validados = validar_eventos(eventos)

    #Quarantine: los que fallaron ante alguna regla, con el motivo propio de c/caso:
    (validados.filter(F.col("motivo").isNotNull())
        .write.mode("overwrite").partitionBy("event_date")
        .parquet(f"{CONFIG['quarantine']}/usage_events"))

    #Silver: los válidos:
    silver = armar_silver(validados.filter(F.col("motivo").isNull()))
    silver.write.mode("overwrite").partitionBy("event_date").parquet(f"{CONFIG['silver']}/usage_events")

    #Gold: recalculo esos dias. overwrite dinamico = solo piso esas particiones, no en otras
    (armar_gold(silver).write.mode("overwrite").partitionBy("usage_date")
        .parquet(f"{CONFIG['gold']}/{CONFIG['tabla_diaria']}"))

    print(f"Lote {lote_id}: {len(fechas)} dias recalculados")

#Leo Bronze como stream (ídem al esquema de la tabla)
bronze_stream = (spark.readStream
    .schema(bronze_eventos.schema)
    .parquet(f"{CONFIG['bronze']}/usage_events"))

query_silver_gold = (bronze_stream.writeStream
    .foreachBatch(procesar_lote)
    .option("checkpointLocation", f"{CONFIG['checkpoints']}/silver_gold_usage_events")
    .trigger(availableNow=True)
    .start())

query_silver_gold.awaitTermination()

#Conteos finales de cada capa, en modo "print_check"
print("Quarantine:", spark.read.parquet(f"{CONFIG['quarantine']}/usage_events").count())
print("Silver:    ", spark.read.parquet(f"{CONFIG['silver']}/usage_events").count())
print("Gold:      ", spark.read.parquet(f"{CONFIG['gold']}/{CONFIG['tabla_diaria']}").count())'''

#SEGUNDA VUELTA PARA EVITAR PROBLEMAS Y TENER QUE CORRER TODO DE NUEVO:
def procesar_lote(lote_df, lote_id):
    #1. Que dias trae este micro-lote
    fechas = [r.event_date for r in lote_df.select("event_date").distinct().collect()]
    if not fechas:
        return

    #2. Leo de Bronze TODOS los eventos de esos dias (no solo los del lote) y los valido
    eventos = (spark.read.parquet(f"{CONFIG['bronze']}/usage_events")
               .filter(F.col("event_date").isin(fechas)))
    validados = validar_eventos(eventos)

    #3. Quarantine: los que fallaron alguna regla, con el motivo.
    #repartition por fecha = 1 archivo por dia (control de archivos chicos)
    (validados.filter(F.col("motivo").isNotNull())
        .repartition("event_date")
        .write.mode("overwrite").partitionBy("event_date")
        .parquet(f"{CONFIG['quarantine']}/usage_events"))

    #4. Silver: los validos, enriquecidos
    silver = armar_silver(validados.filter(F.col("motivo").isNull()))
    (silver.repartition("event_date")
        .write.mode("overwrite").partitionBy("event_date")
        .parquet(f"{CONFIG['silver']}/usage_events"))

    #5. Gold: recalculo esos dias. overwrite dinamico = solo piso esas particiones
    (armar_gold(silver).repartition("usage_date")
        .write.mode("overwrite").partitionBy("usage_date")
        .parquet(f"{CONFIG['gold']}/{CONFIG['tabla_diaria']}"))

    print(f"Lote {lote_id}: {len(fechas)} dias recalculados")

#Leo Bronze como stream (mismo esquema que ya tiene la tabla)
bronze_stream = (spark.readStream
    .schema(bronze_eventos.schema)
    .parquet(f"{CONFIG['bronze']}/usage_events"))

query_silver_gold = (bronze_stream.writeStream
    .foreachBatch(procesar_lote)
    .option("checkpointLocation", f"{CONFIG['checkpoints']}/silver_gold_usage_events")
    .trigger(availableNow=True)
    .start())

query_silver_gold.awaitTermination()

#Conteos finales de cada capa
print("Quarantine:", spark.read.parquet(f"{CONFIG['quarantine']}/usage_events").count())
print("Silver:    ", spark.read.parquet(f"{CONFIG['silver']}/usage_events").count())
print("Gold:      ", spark.read.parquet(f"{CONFIG['gold']}/{CONFIG['tabla_diaria']}").count())

Lote 0: 60 dias recalculados
Quarantine: 2244
Silver:     40956
Gold:       11050


In [12]:
gold_diario = spark.read.parquet(f"{CONFIG['gold']}/{CONFIG['tabla_diaria']}")

#Busco la ultima fecha con datos y tomo los 14 dias hasta esa fecha, como lo indica el ejercicio
fecha_max = gold_diario.agg(F.max("usage_date")).first()[0]
fecha_desde = F.date_sub(F.lit(fecha_max), 13)
print("Ventana de 14 dias: desde", spark.range(1).select(fecha_desde).first()[0], "hasta", fecha_max)

#Sumo el costo de cada servicio en esos 14 dias, por organizacion... y veremos qué sale
top14 = (gold_diario
    .filter(F.col("usage_date") >= fecha_desde)
    .groupBy("org_id", "service")
    .agg(F.round(F.sum("daily_cost_usd"), 4).alias("cost_14d"),
         F.sum("requests").alias("requests_14d"))
    .withColumn("window_end", F.lit(fecha_max)))

top14.coalesce(1).write.mode("overwrite").parquet(f"{CONFIG['gold']}/{CONFIG['tabla_top14']}")
#veo qué pasó.
print("Filas en top14:", top14.count())
top14.filter(F.col("org_id") == "org_0lvsnujz").orderBy(F.desc("cost_14d")).show()

Ventana de 14 dias: desde 2025-08-18 hasta 2025-08-31
Filas en top14: 262
+------------+---------+--------+------------+----------+
|      org_id|  service|cost_14d|requests_14d|window_end|
+------------+---------+--------+------------+----------+
|org_0lvsnujz|analytics|406.2605|      2595.0|2025-08-31|
|org_0lvsnujz| database|148.3036|      2622.0|2025-08-31|
|org_0lvsnujz|  storage| 61.1472|      3012.0|2025-08-31|
+------------+---------+--------+------------+----------+



In [13]:
from google.colab import userdata
from astrapy import DataAPIClient
from astrapy.info import CreateTableDefinition, ColumnType
from astrapy.constants import SortMode

#Leo las credenciales desde Colab Secrets, donde las creé:
token = userdata.get("ASTRA_DB_APPLICATION_TOKEN")
endpoint = userdata.get("ASTRA_DB_API_ENDPOINT")

cliente = DataAPIClient()
db = cliente.get_database(endpoint, token=token, keyspace=CONFIG["astra_keyspace"])

#Tabla 1 (consulta 1): costos y requests por org y servicio en un rango de fechas.
#Particion: org_id. Clustering: usage_date y service, asi puedo pedir un rango de fechas de una org
#sin que esto me represente algún problema:
def_diaria = (CreateTableDefinition.builder()
    .add_column("org_id", ColumnType.TEXT)
    .add_column("usage_date", ColumnType.DATE)
    .add_column("service", ColumnType.TEXT)
    .add_column("org_name", ColumnType.TEXT)
    .add_column("daily_cost_usd", ColumnType.DOUBLE)
    .add_column("requests", ColumnType.DOUBLE)
    .add_column("cpu_hours", ColumnType.DOUBLE)
    .add_column("storage_gb_hours", ColumnType.DOUBLE)
    .add_column("genai_tokens", ColumnType.BIGINT)
    .add_column("carbon_kg", ColumnType.DOUBLE)
    .add_column("eventos", ColumnType.INT)
    .add_partition_by(["org_id"])
    .add_partition_sort({"usage_date": SortMode.ASCENDING, "service": SortMode.ASCENDING})
    .build())

#Tabla 2 (consulta 2): top-N de servicios por costo en 14 dias.
#Clustering por cost_14d DESC: Cassandra ya me los guarda ordenados de mayor a menor, así que
#en estas líneas omito ese paso:
def_top14 = (CreateTableDefinition.builder()
    .add_column("org_id", ColumnType.TEXT)
    .add_column("cost_14d", ColumnType.DOUBLE)
    .add_column("service", ColumnType.TEXT)
    .add_column("requests_14d", ColumnType.DOUBLE)
    .add_column("window_end", ColumnType.DATE)
    .add_partition_by(["org_id"])
    .add_partition_sort({"cost_14d": SortMode.DESCENDING, "service": SortMode.ASCENDING})
    .build())

tabla_diaria = db.create_table(CONFIG["tabla_diaria"], definition=def_diaria, if_not_exists=True)
tabla_top14 = db.create_table(CONFIG["tabla_top14"], definition=def_top14, if_not_exists=True)
#check:
print("Tablas en Astra:", db.list_table_names())

Tablas en Astra: ['org_daily_usage_by_service', 'top_services_14d']


In [16]:
#otra def, para no perder la costumbre.
#CUIDADO: tira error si no se actualiza Astra_DB a tiempo. Si hay error, se ejecuta de neuvo:
def a_filas(df):
    #Paso el DataFrame de Spark a una lista de diccionarios y así enviarlo a Astra_DB
    #Las fechas van como texto 'aaaa-mm-dd', que es el formato que me acepta la API
    filas = []
    for r in df.collect():
        d = r.asDict()
        for k, v in d.items():
            if hasattr(v, "isoformat"):
                d[k] = v.isoformat()
        filas.append(d)
    return filas

columnas_diaria = ["org_id", "usage_date", "service", "org_name", "daily_cost_usd", "requests",
                   "cpu_hours", "storage_gb_hours", "genai_tokens", "carbon_kg", "eventos"]
filas_diaria = a_filas(gold_diario.select(columnas_diaria))

filas_top14 = a_filas(spark.read.parquet(f"{CONFIG['gold']}/{CONFIG['tabla_top14']}"))

#insert_many en tablas funciona como UPSERT: si la clave primaria ya existe, la pisa.
#Por eso, si vuelvo a cargar, no se duplican filas (idempotencia)
r1 = tabla_diaria.insert_many(filas_diaria, chunk_size=50, concurrency=10)
r2 = tabla_top14.insert_many(filas_top14, chunk_size=50, concurrency=10)
#más chequeos:
print("Enviadas a", CONFIG["tabla_diaria"], ":", len(filas_diaria))
print("Enviadas a", CONFIG["tabla_top14"], ":", len(filas_top14))

Enviadas a org_daily_usage_by_service : 11050
Enviadas a top_services_14d : 262


In [17]:
import pandas as pd #por más que sea... nunca falta (ni falla) pandas.

org = "org_0lvsnujz"

#Consulta 1: costos y requests diarios de una organizacion por servicio, con rango de fechas.
#Funciona rapido porque filtro por la clave de particion (org_id) y un rango del clustering (usage_date)
c1 = tabla_diaria.find(
    {"org_id": org, "usage_date": {"$gte": "2025-08-01", "$lte": "2025-08-07"}},
    projection={"usage_date": True, "service": True, "daily_cost_usd": True, "requests": True})
print("Consulta 1 -", org, "del 01/08 al 07/08/2025")
print(pd.DataFrame([{k: str(v) if k == "usage_date" else v for k, v in f.items()} for f in c1]))

#Consulta 2: top-3 servicios por costo acumulado en los ultimos 14 dias.
#No ordeno nada: la tabla ya guarda los servicios de mayor a menor costo (cost_14d DESC)
c2 = tabla_top14.find({"org_id": org}, limit=3,
                      projection={"service": True, "cost_14d": True, "requests_14d": True})
print("\nConsulta 2 - Top 3 servicios por costo en 14 dias -", org)
print(pd.DataFrame(list(c2)))

Consulta 1 - org_0lvsnujz del 01/08 al 07/08/2025
    usage_date    service  daily_cost_usd  requests
0   2025-08-01  analytics         24.6844     363.0
1   2025-08-01   database         13.6787     241.0
2   2025-08-01    storage          8.4007     399.0
3   2025-08-02  analytics         33.5652     374.0
4   2025-08-02   database          0.8243       NaN
5   2025-08-02    storage          2.2995     109.0
6   2025-08-03  analytics          5.1203     124.0
7   2025-08-03    storage          6.4308     355.0
8   2025-08-04  analytics         30.2652     439.0
9   2025-08-04   database          1.5956       NaN
10  2025-08-04    storage          7.0998     153.0
11  2025-08-05  analytics         13.1185     207.0
12  2025-08-05   database          6.9888     115.0
13  2025-08-05    storage          4.1750     223.0
14  2025-08-06  analytics         17.2325     242.0
15  2025-08-06   database         12.1056     233.0
16  2025-08-06    storage          7.6186     352.0
17  2025-08-07

In [18]:
#pruebo la idempotencia que cita la consigna y lo aprendido en clases
import shutil

def conteos():
    return {
        "bronze": spark.read.parquet(f"{CONFIG['bronze']}/usage_events").count(),
        "quarantine": spark.read.parquet(f"{CONFIG['quarantine']}/usage_events").count(),
        "silver": spark.read.parquet(f"{CONFIG['silver']}/usage_events").count(),
        "gold": spark.read.parquet(f"{CONFIG['gold']}/{CONFIG['tabla_diaria']}").count(),
        "astra_org_ejemplo": len(list(tabla_diaria.find({"org_id": org}, projection={"org_id": True})))
    }

antes = conteos()

#Prueba 1: vuelvo a correr el stream de Bronze con el mismo checkpoint, sin archivos nuevos
q = (eventos_bronze.writeStream.format("parquet").outputMode("append")
     .partitionBy("event_date")
     .option("path", f"{CONFIG['bronze']}/usage_events")
     .option("checkpointLocation", f"{CONFIG['checkpoints']}/bronze_usage_events")
     .trigger(availableNow=True).start())
q.awaitTermination()
print("Prueba 1 - filas nuevas leidas por el stream:", sum(p["numInputRows"] for p in q.recentProgress))

#Prueba 2: simulo que llega de nuevo un archivo ya procesado (re-entrega, pero con otro nombre)
origen = f"{CONFIG['landing']}/usage_events_stream/events_part_0000.jsonl"
copia = f"{CONFIG['landing']}/usage_events_stream/events_part_9999_reenvio.jsonl"
shutil.copy(origen, copia)

q = (eventos_bronze.writeStream.format("parquet").outputMode("append")
     .partitionBy("event_date")
     .option("path", f"{CONFIG['bronze']}/usage_events")
     .option("checkpointLocation", f"{CONFIG['checkpoints']}/bronze_usage_events")
     .trigger(availableNow=True).start())
q.awaitTermination()
print("Prueba 2 - filas leidas del archivo reenviado:", sum(p["numInputRows"] for p in q.recentProgress))

#Corro de nuevo la Query 2 (Silver/Gold) por si algo de más hubiera entrado
q2 = (bronze_stream.writeStream.foreachBatch(procesar_lote)
      .option("checkpointLocation", f"{CONFIG['checkpoints']}/silver_gold_usage_events")
      .trigger(availableNow=True).start())
q2.awaitTermination()

#Recargo Gold en Astra (upsert): no tiene que cambiar la cantidad de filas
tabla_diaria.insert_many(a_filas(gold_diario.select(columnas_diaria)), chunk_size=50, concurrency=10)

#reviso: si todo está Ok en la tabla, salió bien:
despues = conteos()
print("\nCapa                antes    despues")
for k in antes:
    print(f"{k:18s} {antes[k]:7d}  {despues[k]:7d}  {'OK' if antes[k] == despues[k] else 'DISTINTO'}")

Prueba 1 - filas nuevas leidas por el stream: 0
Prueba 2 - filas leidas del archivo reenviado: 360

Capa                antes    despues
bronze               43200    43200  OK
quarantine            2244     2244  OK
silver               40956    40956  OK
gold                 11050    11050  OK
astra_org_ejemplo      156      156  OK


In [19]:
import subprocess
from datetime import datetime

EVID = "/content/evidence"
os.makedirs(EVID, exist_ok=True)

#Conteos por capa y resultado de la idempotencia
with open(f"{EVID}/conteos_e_idempotencia.txt", "w") as f:
    f.write(f"Ejecucion: {datetime.now()}\n\n")
    f.write("Batch CSV -> Bronze (leidas / en bronze)\n")
    for t, (l, b) in conteos_bronze.items():
        f.write(f"  {t:20s} {l:6d} {b:6d}\n")
    f.write("\nIdempotencia (antes / despues)\n")
    for k in antes:
        f.write(f"  {k:18s} {antes[k]:7d} {despues[k]:7d}\n")

#Resumen de quarantine por motivo y una muestra de solamente 20 registros
q_df = spark.read.parquet(f"{CONFIG['quarantine']}/usage_events")
q_df.groupBy("motivo").count().toPandas().to_csv(f"{EVID}/quarantine_por_motivo.csv", index=False)
(q_df.select("event_id", "event_date", "org_id", "service", "metric", "value", "unit",
             "cost_usd_increment", "motivo", "source_file")
     .limit(20).toPandas().to_csv(f"{EVID}/quarantine_muestra.csv", index=False))

#Rutas y tamaños del Data Lake, la perfeformance del control de archivos
salida = subprocess.run("du -sh /content/datalake/*/* && echo && "
                        "find /content/datalake -name '*.parquet' | wc -l",
                        shell=True, capture_output=True, text=True).stdout
with open(f"{EVID}/rutas_y_tamanios.txt", "w") as f:
    f.write(salida)

#Resultado de las 2 consultas contrra Astra
with open(f"{EVID}/consultas_astra.txt", "w") as f:
    c1 = tabla_diaria.find({"org_id": org, "usage_date": {"$gte": "2025-08-01", "$lte": "2025-08-07"}},
                           projection={"usage_date": True, "service": True, "daily_cost_usd": True, "requests": True})
    f.write(f"Consulta 1 - {org} del 2025-08-01 al 2025-08-07\n")
    f.write(pd.DataFrame([{k: str(v) for k, v in r.items()} for r in c1]).to_string() + "\n\n")
    c2 = tabla_top14.find({"org_id": org}, limit=3, projection={"service": True, "cost_14d": True})
    f.write(f"Consulta 2 - Top 3 servicios por costo en 14 dias - {org}\n")
    f.write(pd.DataFrame(list(c2)).to_string() + "\n")

#Copio tambien el config.json y comprimo todo para descargarlo y dejarlo listo
shutil.copy("/content/config/config.json", f"{EVID}/config_usado.json")
shutil.make_archive("/content/evidence", "zip", EVID)
print(open(f"{EVID}/rutas_y_tamanios.txt").read())
print("Archivos de evidencia:", os.listdir(EVID))

3.0M	/content/datalake/_checkpoints/bronze_usage_events
180K	/content/datalake/_checkpoints/silver_gold_usage_events
52K	/content/datalake/bronze/billing_monthly
20K	/content/datalake/bronze/customers_orgs
36K	/content/datalake/bronze/marketing_touches
16K	/content/datalake/bronze/nps_surveys
24K	/content/datalake/bronze/resources
32K	/content/datalake/bronze/support_tickets
12M	/content/datalake/bronze/usage_events
36K	/content/datalake/bronze/users
1.2M	/content/datalake/gold/org_daily_usage_by_service
20K	/content/datalake/gold/top_services_14d
16K	/content/datalake/landing/billing_monthly.csv
8.0K	/content/datalake/landing/customers_orgs.csv
100K	/content/datalake/landing/marketing_touches.csv
4.0K	/content/datalake/landing/nps_surveys.csv
36K	/content/datalake/landing/resources.csv
72K	/content/datalake/landing/support_tickets.csv
13M	/content/datalake/landing/usage_events_stream
72K	/content/datalake/landing/users.csv
1.2M	/content/datalake/quarantine/usage_events
3.3M	/content/d